In [2]:
import json, os, pickle
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd

import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
import qiskit.circuit.random
import torch, random
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn
from qiskit.qpy import load

import numpy as np
import json, os, pickle
from tqdm import tqdm
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from qiskit.qasm2 import dump

from io import StringIO

from qiskit import QuantumCircuit

import sys
sys.path.append('../tutorials/')
from mlp import encode_data, encode_data_v2_ecr

In [13]:
def load_circuits(data_dir, json_file_dir):

    circuits = []
    noisy_expected_values = []
    noiseless_expected_values = []

    with open(json_file_dir) as json_file:
        json_data = json.load(json_file)

    circuit_names = list(json_data.keys())
    # print(circuit_names)

    for circuit_name in tqdm(circuit_names, leave=True):

        file_path = os.path.join(data_dir, circuit_name)
        try:
            with open(file_path, "rb") as f:
                circuit = load(f)  # returns a list of QuantumCircuit objects
                if len(json_data[circuit_name]["z_noisy"][:5]) != 5:
                    continue
                else:
                    circuits.append(circuit[0])
                    noisy_expected_values.append(json_data[circuit_name]["z_noisy"][:5])
                    noiseless_expected_values.append(json_data[circuit_name]["z_ideal"][:5])
        except Exception as e:
            print(f"⚠️ Error loading {file}: {e}")
            continue

    return circuits, noisy_expected_values, noiseless_expected_values


def load_circuits_real_data(data_dir, real_json_file_dir):

    circuits = []
    noisy_expected_values = []
    noiseless_expected_values = []

    all_ciruits = os.listdir(data_dir)

    # with open(json_file_dir) as json_file:
    #     json_data = json.load(json_file)

    # circuit_names = list(json_data.keys())
    # print(circuit_names)

    for circuit_name in tqdm(all_ciruits, leave=True):

        real_file_path = os.path.join(real_json_file_dir,f"{circuit_name}__hardware_shots.json")
        sim_file_path = os.path.join(real_json_file_dir,f"{circuit_name}__ideal_shots.json")

        try:
            with open(real_file_path) as real_json_file:
                real_data_loaded = json.load(real_json_file)
            with open(sim_file_path) as ideal_json_file:
                ideal_data_loaded = json.load(ideal_json_file)
        except Exception as e:
            print(f"⚠️ Error loading: {e}")
            continue
        
        # file_path = os.path.join(data_dir, circuit_name)
        try:
            with open(file_path, "rb") as f:
                circuit = load(f)  # returns a list of QuantumCircuit objects
                if len(real_data_loaded["z"][:5]) != 5:
                    continue
                else:
                    circuits.append(circuit[0])
                    noisy_expected_values.append(real_file_path["z"][:5])
                    noiseless_expected_values.append(ideal_data_loaded["z"][:5])
        except Exception as e:
            # print(f"⚠️ Error loading {file}: {e}")
            continue

    return circuits, noisy_expected_values, noiseless_expected_values


In [4]:

data_dir = "../../../andrew/ExecutionResults/StoredCircuits/"

all_data = os.listdir(data_dir)

In [5]:
all_data

['9442e55b-2c38-4287-85c0-61af0dcc7dbc.qpy',
 'c5613591-338a-4306-aa75-735c9385c8e4.qpy',
 'ea11a710-d03d-4d65-be29-0c7a5949fee2.qpy',
 'ac78e173-fa48-47fb-980b-1d2ba9628fd3.qpy',
 '8d0af168-21d5-4b59-8495-183145b19d61.qpy',
 'a31d466e-951d-422d-ba34-90ea0e6512bb.qpy',
 '331b4ae6-ecbf-411e-86e7-295ae38b748d.qpy',
 'b0220501-b363-4e34-93eb-6223ec809aab.qpy',
 'b360114e-6ebf-480d-9d81-ccc92613ebce.qpy',
 '22aa0a2c-7f6b-49c9-b66d-1e7f14ec4543.qpy',
 '6acb4ed0-8731-453a-b767-3cb21a164721.qpy',
 '1fe5fcbd-c1ee-4c10-b472-56ee03337215.qpy',
 '78ad8263-94ca-4153-9716-6e595d7fce9d.qpy',
 'ff5140d6-c452-4630-8bf2-8d26cc1a2fbe.qpy',
 '81917f2f-c962-48f9-8a11-c8afea0053d1.qpy',
 '5038af46-d7be-4859-bbd9-5beb9ce13fde.qpy',
 '0c1b3603-4e2a-47a1-b923-78eb694970d5.qpy',
 'f2d2ede1-a045-4d00-86c2-70045436c87f.qpy',
 '7b0c0c5a-b9b4-4cff-82b5-049721bb8622.qpy',
 'fa57fc65-9c56-48d6-9395-10db2e5258d4.qpy',
 'ae30e4a4-65d4-43b9-a6da-f49e9f679f5a.qpy',
 '3bfcbdea-f067-4b36-b942-7d4bfc0e5a9b.qpy',
 '1ae65651

In [14]:
data_dir = "../../../andrew/ExecutionResults/StoredCircuits/"
json_file_dir = "./real-qpu-submissions/real_z_expectations_shot_data/"

circuits, noisy_expected_values, noiseless_expected_values = load_circuits_real_data(data_dir, json_file_dir)

  3%|███▍                                                                                                   | 236/7004 [00:00<00:06, 1054.83it/s]

⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/5f504181-09fb-4b80-811c-498f45c85525.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/9b8bb407-860e-4300-9c49-8c65d98b419a.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/943456b9-b16d-415e-8aa4-1039d45fc8ad.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/cc9aefec-acdf-4bdd-9388-d4ef805d0e1c.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/726a985f-5921-427f-94e6-188efd5db739.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/5cf44548-36f6-473a-8089-95c9ccb

 20%|████████████████████▎                                                                                 | 1397/7004 [00:00<00:01, 4188.00it/s]

⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/10dab9a3-4be4-4dc2-ab3d-aa84d40343d5.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/84c51ae0-cc70-471d-b55c-8bbdb4570a96.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/686246f6-39d3-4e0e-aec4-9861ba4ea423.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/a40581b6-8699-4f1f-8f95-e4c6ac7d1f94.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/e8679b25-08a3-46fc-8568-794fd6cc814c.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/92ceb40e-2fb9-4171-b7cf-fe1fd3e

 39%|███████████████████████████████████████▎                                                              | 2700/7004 [00:00<00:00, 5515.49it/s]

...

⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/5d80fc7f-0dd4-44b0-852f-daf396dd50fa.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/2ee371f0-f09d-42e2-919f-bca92fc3447c.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/f974690f-badd-4ed3-af42-ba7df14cf1c1.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/19dcf9bc-0bcd-4a8a-ab69-8ae5742b61b5.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/50e3b8f0-7d8d-4aca-bc1a-c8172a4b79f3.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/96b50c38-3634-45a4-8686-3d9b40a

 83%|█████████████████████████████████████████████████████████████████████████████████████                 | 5839/7004 [00:01<00:00, 4474.29it/s]

⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/7e465b08-70bc-4fd6-b719-c2f5524ded05.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/d0ed4d6d-2476-43d2-9d13-c7b61d51198d.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/3ad14f00-b62b-456a-9bb5-af0aebd7ee68.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/4b97db85-9de8-4c50-bff3-00bf0a4edf3a.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/09552d06-06d5-4926-81d1-41917fcf54b7.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/999e2387-21b6-4664-9047-61ac442

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 7004/7004 [00:01<00:00, 4481.74it/s]

⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/7eadf9da-78c7-4b1c-8853-10df1bbea808.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/07042514-2afb-47f6-804f-5d0f31b7536d.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/5a3b5b4b-b694-4b6f-9c0d-6d2518a259a2.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/de207be7-4381-4c2d-8941-5b91a2bf153f.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/28d99f72-6df3-4afd-a283-09cc2d3621e9.qpy__hardware_shots.json'
⚠️ Error loading: [Errno 2] No such file or directory: './real-qpu-submissions/real_z_expectations_shot_data/3200c600-9527-4954-bf37-38f2f5e

In [15]:
len(circuits)

0

In [17]:
def count_gates_by_rotation_angle(circuit, bin_size):
    angles = []
    for instr, qargs, cargs in circuit.data:
        if instr.name in ['rx', 'ry', 'rz'] and len(qargs) == 1:
            angles += [float(instr.params[0])]
    bin_edges = np.arange(-2 * np.pi, 2 * np.pi + bin_size, bin_size)
    counts, _ = np.histogram(angles, bins=bin_edges)
    bin_labels = [f"{left:.2f} to {right:.2f}" for left, right in zip(bin_edges[:-1], bin_edges[1:])]
    angle_bins = {label: count for label, count in zip(bin_labels, counts)}
    return list(angle_bins.values())


def recursive_dict_loop(my_dict, parent_key=None, out=None, target_key1=None, target_key2=None):
    if out is None: out = []

    for key, val in my_dict.items():
        if isinstance(val, dict):
            recursive_dict_loop(val, key, out, target_key1, target_key2)
        else:
            if parent_key and target_key1 in str(parent_key) and key == target_key2:
                out += [val]
    return out or 0.


def encode_data_v2_ecr_CLIP_encoder(circuits, 
                                    ideal_exp_vals, 
                                    noisy_exp_vals, 
                                    obs_size, 
                                    tokenizer,
                                    text_model,
                                    device,
                                    max_len = 77,
                                    meas_bases=None, 
                                    two_q_gate='ecr'):
    
    if isinstance(noisy_exp_vals[0], list) and len(noisy_exp_vals[0]) == 1:
        noisy_exp_vals = [x[0] for x in noisy_exp_vals]

    if meas_bases is None:
        meas_bases = [[]]

    gates_set = [two_q_gate] + ['sx', 'x', 'id', 'rz']

    vec = []

    bin_size = 0.025 * np.pi
    num_angle_bins = int(np.ceil(4 * np.pi / bin_size))

    X = torch.zeros([len(circuits), 512 + len(vec) + len(gates_set) + num_angle_bins + obs_size + len(meas_bases[0])]) #512 for CLIP Embedding

    embedding_slice = slice(0,512)
    vec_slice = slice(512, 512+len(vec))
    gate_counts_slice = slice(512+len(vec), 512+len(vec)+len(gates_set))
    angle_bins_slice = slice(512+len(vec)+len(gates_set), 512+len(vec)+len(gates_set)+num_angle_bins)
    exp_val_slice = slice(512+len(vec)+len(gates_set)+num_angle_bins, 512+len(vec)+len(gates_set)+num_angle_bins+obs_size)
    meas_basis_slice = slice(512+len(vec)+len(gates_set)+num_angle_bins+obs_size, len(X[0]))

    # X[:, vec_slice] = vec[None, :]

    for i, circ in enumerate(tqdm(circuits)):
        qasm_buffer = StringIO()
        dump(circ, qasm_buffer)
        qasm_code = qasm_buffer.getvalue()
        
        circuit_qasm = qasm_code

        tokens = tokenizer(circuit_qasm, return_tensors="pt", truncation=False, padding=False)
        input_ids = tokens["input_ids"][0]  # remove batch dimension
        
        chunks = [input_ids[i:i + max_len] for i in range(0, len(input_ids), max_len)]
        
        # Encode each chunk and collect pooled outputs
        embeddings = []
        
        for chunk in chunks:
            chunk = chunk.unsqueeze(0).to(device)
            with torch.no_grad():
                output = text_model(input_ids=chunk)
                pooled = output.pooler_output  # shape: (1, hidden_dim)
            embeddings.append(pooled.cpu())  # keep CPU to save GPU memory
        
        # Combine embeddings (mean pooling)
        final_embedding = torch.mean(torch.stack(embeddings), dim=0)

        
        X[i, embedding_slice] = torch.tensor(final_embedding)

    
    for i, circ in enumerate(circuits):
        gate_counts_all = circ.count_ops()
        X[i, gate_counts_slice] = torch.tensor(
            [gate_counts_all.get(key, 0) for key in gates_set]
        ) * 0.01  # put it in the same order of magnitude as the expectation values

    for i, circ in enumerate(circuits):
        gate_counts = count_gates_by_rotation_angle(circ, bin_size)
        X[i, angle_bins_slice] = torch.tensor(gate_counts) * 0.01  # put it in the same order of magnitude as the expectation values

        if obs_size > 1: assert len(noisy_exp_vals[i]) == obs_size
        elif obs_size == 1: assert isinstance(noisy_exp_vals[i], float)

        X[i, exp_val_slice] = torch.tensor(noisy_exp_vals[i])

    if meas_bases != [[]]:
        assert len(meas_bases) == len(circuits)
        for i, basis in enumerate(meas_bases):
            X[i, meas_basis_slice] = torch.tensor(basis)

    y = torch.tensor(ideal_exp_vals, dtype=torch.float32)

    return X, y

In [30]:
num_circ_per_step = 50
k = train_test_split = 40
train_circuits = []
train_ideal_vals = []
train_noisy_vals = []
test_circuits = []
test_ideal_vals = []
test_noisy_vals = []
test_Js = []
for start_each_step in list(range(len(circuits))[::num_circ_per_step]):
    train_circuits += circuits[start_each_step:start_each_step+k]
    train_ideal_vals += noiseless_expected_values[start_each_step:start_each_step+k]
    train_noisy_vals += noisy_expected_values[start_each_step:start_each_step+k]
    test_circuits += circuits[start_each_step+k:start_each_step+num_circ_per_step]
    test_ideal_vals += noiseless_expected_values[start_each_step+k:start_each_step+num_circ_per_step]
    test_noisy_vals += noisy_expected_values[start_each_step+k:start_each_step+num_circ_per_step]
    # test_Js += Js[start_each_step+k:start_each_step+num_circ_per_step]

In [31]:
print(len(train_circuits), len(train_ideal_vals), len(train_noisy_vals))
print(len(test_circuits), len(test_ideal_vals), len(test_noisy_vals))

5233 5233 5233
1300 1300 1300


In [32]:
normal_X_train, normal_y_train = encode_data_v2_ecr(train_circuits, train_ideal_vals, train_noisy_vals, obs_size=5)
normal_X_test, normal_y_test = encode_data_v2_ecr(test_circuits, test_ideal_vals, test_noisy_vals, obs_size=5)

In [33]:
print(normal_X_train.shape, normal_y_train.shape)
print(normal_X_test.shape, normal_y_test.shape)

torch.Size([5233, 170]) torch.Size([5233, 5])
torch.Size([1300, 170]) torch.Size([1300, 5])


In [9]:
from transformers import CLIPTokenizer, CLIPTextModel
import torch
from tqdm import tqdm
import time

In [10]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load tokenizer and model
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
text_model.eval()

Using device: cuda


CLIPTextModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

In [34]:
X_train, y_train = encode_data_v2_ecr_CLIP_encoder(train_circuits, 
                                                   train_ideal_vals, 
                                                   train_noisy_vals, 
                                                   tokenizer=tokenizer,
                                                   text_model=text_model,
                                                   device=device,
                                                   obs_size=5)

  0%|                                                                                                                   | 0/5233 [00:00<?, ?it/s]/tmp/ipykernel_2284949/4169997577.py:86: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X[i, embedding_slice] = torch.tensor(final_embedding)
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 5233/5233 [03:39<00:00, 23.87it/s]
/tmp/ipykernel_2284949/4169997577.py:3: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for instr, qargs, cargs in circuit.data:


In [35]:
torch.save(X_train, 'CLIP_Average_X_train_Andrew.pt')
torch.save(y_train, 'CLIP_Average_y_train_Andrew.pt')



In [36]:
X_test, y_test = encode_data_v2_ecr_CLIP_encoder(test_circuits, 
                                                 test_ideal_vals, 
                                                 test_noisy_vals, 
                                                 tokenizer=tokenizer,
                                                 text_model=text_model,
                                                 device=device,
                                                 obs_size=5)

torch.save(X_test, 'CLIP_Average_X_test_Andrew.pt')
torch.save(y_test, 'CLIP_Average_y_test_Andrew.pt')

  0%|                                                                                                                   | 0/1300 [00:00<?, ?it/s]/tmp/ipykernel_2284949/4169997577.py:86: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X[i, embedding_slice] = torch.tensor(final_embedding)
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1300/1300 [00:55<00:00, 23.24it/s]
/tmp/ipykernel_2284949/4169997577.py:3: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for instr, qargs, cargs in circuit.data:
